# routes/init

> Router assembly — creates all sub-routers, populates URL bundle, returns unified interface

In [ ]:
#| default_exp routes.init

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, List, Tuple, Callable

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls
from cjm_transcription_source_select.services.source_select import SourceSelectService
from cjm_transcription_source_select.routes.browser import init_browser_router
from cjm_transcription_source_select.routes.selection import init_selection_router
from cjm_transcription_source_select.routes.preview import init_preview_router
from cjm_transcription_source_select.routes.verify import init_verify_router

## Router Assembly

Creates all sub-routers, populates the URL bundle, and returns a unified interface.

Follows the same 4-phase pattern as `cjm-transcript-source-select`:
1. Create empty URL bundle
2. Initialize focused sub-routers (each receives the mutable URL bundle)
3. Populate URLs using `.to()` on route functions
4. Merge and return `(routers, urls, merged_routes)`

In [ ]:
#| export
def init_source_select_routers(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    browser_config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    service: SourceSelectService,  # Source select service (FFmpeg plugin)
    home_path: str = "",  # Home directory for nav buttons
    prefix: str = "/source_select",  # Base route prefix
) -> Tuple[List[APIRouter], SourceSelectUrls, Dict[str, Callable]]:  # (routers, urls, routes)
    """Initialize all source selection routers and populate URL bundle."""
    _home = home_path or provider.get_home_path()

    # Phase 1: Create empty URL bundle
    urls = SourceSelectUrls()

    # Phase 2: Initialize sub-routers
    browser_router, browser_routes = init_browser_router(
        state_store=state_store,
        provider=provider,
        config=browser_config,
        workflow_id=workflow_id,
        urls=urls,
        home_path=_home,
        prefix=f"{prefix}/browser",
    )

    selection_router, selection_routes = init_selection_router(
        state_store=state_store,
        provider=provider,
        config=browser_config,
        workflow_id=workflow_id,
        urls=urls,
        home_path=_home,
        prefix=f"{prefix}/selection",
    )

    preview_router, preview_routes = init_preview_router(
        state_store=state_store,
        workflow_id=workflow_id,
        urls=urls,
        prefix=f"{prefix}/preview",
    )

    verify_router, verify_routes = init_verify_router(
        state_store=state_store,
        workflow_id=workflow_id,
        urls=urls,
        service=service,
        prefix=f"{prefix}/verify",
    )

    # Phase 3: Populate URL bundle
    urls.navigate = browser_routes["navigate"].to()
    urls.select = browser_routes["select"].to()
    urls.toggle_view = browser_routes["toggle_view"].to()
    urls.change_sort = browser_routes["change_sort"].to()
    urls.remove = selection_routes["remove"].to()
    urls.reorder = selection_routes["reorder"].to()
    urls.clear = selection_routes["clear"].to()
    urls.preview = preview_routes["preview"].to()
    urls.media_src = preview_routes["media_src"].to()
    urls.verify = verify_routes["verify"].to()

    # Phase 4: Merge and return
    merged_routes = {
        **browser_routes,
        **selection_routes,
        **preview_routes,
        **verify_routes,
    }
    routers = [browser_router, selection_router, preview_router, verify_router]

    return routers, urls, merged_routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()